# Module 1 — Transformer Internals (exercises)

Homework-style. Fill every `# TODO`, then run the test cell under it — the `assert`s grade you.
Core exercises are **pure numpy** and run offline. Cells marked **OPTIONAL** need torch / nanoGPT.

Rules: no peeking at `torch.nn.MultiheadAttention` until Exercise 6. Derive shapes yourself.

Companion reading: `workbook.md` in this folder.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def approx(a, b, tol=1e-6):
    return np.allclose(a, b, atol=tol)

## Exercise 1 — Numerically stable softmax
Implement softmax over a chosen axis. Subtract the max before exponentiating (why? overflow).
It must work on a 1-D vector and on the last axis of a batched 3-D tensor.

In [ ]:
def softmax(x, axis=-1):
    # TODO: subtract max along `axis` (keepdims), exponentiate, divide by the sum along `axis`.
    raise NotImplementedError

# --- tests ---
v = np.array([1.0, 2.0, 3.0])
p = softmax(v)
assert approx(p.sum(), 1.0), 'rows must sum to 1'
assert np.argmax(p) == 2, 'largest logit -> largest prob'
big = np.array([1000.0, 1001.0, 1002.0])  # would overflow without the max trick
assert np.isfinite(softmax(big)).all(), 'must be numerically stable'
batched = rng.normal(size=(2, 4, 5))
assert approx(softmax(batched, axis=-1).sum(axis=-1), np.ones((2, 4))), 'axis handling'
print('Exercise 1 passed.')

## Exercise 2 — Scaled dot-product attention
`Attention(Q,K,V) = softmax(QKᵀ / √d_k) V`. Shapes: Q,K,V are `(..., seq, d_k)`.
Support an optional additive mask (use `-inf` where disallowed, *before* softmax).
Return `(output, weights)` so you can inspect what attended to what.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    # TODO:
    #   d_k = Q.shape[-1]
    #   scores = Q @ K.swapaxes(-1, -2) / sqrt(d_k)
    #   if mask is not None: scores = where(mask, scores, -inf)   # mask True = keep
    #   weights = softmax(scores, axis=-1)
    #   return weights @ V, weights
    raise NotImplementedError

# --- tests ---
seq, d_k = 4, 8
Q = rng.normal(size=(seq, d_k)); K = rng.normal(size=(seq, d_k)); V = rng.normal(size=(seq, d_k))
out, w = scaled_dot_product_attention(Q, K, V)
assert out.shape == (seq, d_k), 'output shape'
assert approx(w.sum(axis=-1), np.ones(seq)), 'each query distributes weight=1 over keys'
# If all keys identical, every query returns that same value.
Ksame = np.ones((seq, d_k)); Vsame = np.tile(np.arange(d_k), (seq, 1)).astype(float)
out2, _ = scaled_dot_product_attention(Q, Ksame, Vsame)
assert approx(out2, Vsame[0]), 'uniform keys -> mean of (identical) values'
print('Exercise 2 passed.')

## Exercise 3 — Multi-head self-attention
Project x into Q,K,V; split into `h` heads of size `d_model/h`; attend per head; concat; output-project.
Implement as a small class. Use the weight matrices provided so the test is deterministic.

In [ ]:
class MultiHeadSelfAttention:
    def __init__(self, d_model, n_heads, Wq, Wk, Wv, Wo):
        assert d_model % n_heads == 0
        self.d_model, self.h = d_model, n_heads
        self.d_head = d_model // n_heads
        self.Wq, self.Wk, self.Wv, self.Wo = Wq, Wk, Wv, Wo

    def _split(self, x):
        # (seq, d_model) -> (h, seq, d_head)
        seq = x.shape[0]
        return x.reshape(seq, self.h, self.d_head).transpose(1, 0, 2)

    def __call__(self, x, mask=None):
        # TODO:
        #   Q,K,V = x@Wq, x@Wk, x@Wv      # each (seq, d_model)
        #   split each into heads -> (h, seq, d_head)
        #   run scaled_dot_product_attention per head (it broadcasts over the leading h axis)
        #   concat heads back -> (seq, d_model), then @ Wo
        raise NotImplementedError

# --- tests ---
d_model, h, seq = 16, 4, 5
Wq, Wk, Wv, Wo = [rng.normal(size=(d_model, d_model)) for _ in range(4)]
mha = MultiHeadSelfAttention(d_model, h, Wq, Wk, Wv, Wo)
x = rng.normal(size=(seq, d_model))
y = mha(x)
assert y.shape == (seq, d_model), 'shape preserved'
assert np.isfinite(y).all()
print('Exercise 3 passed.')

## Exercise 4 — Sinusoidal positional encoding
`PE(pos,2i)=sin(pos/10000^(2i/d))`, `PE(pos,2i+1)=cos(...)`. Build a `(seq, d)` matrix.
Sanity: each row has the same L2 norm geometry; nearby positions are more similar than far ones.

In [ ]:
def positional_encoding(seq, d):
    # TODO: build (seq, d). pos = arange(seq)[:,None]; i pairs use 10000**(2i/d).
    raise NotImplementedError

# --- tests ---
PE = positional_encoding(20, 16)
assert PE.shape == (20, 16)
assert -1.0001 <= PE.min() and PE.max() <= 1.0001, 'sin/cos bounded'
# adjacent positions more similar than distant ones (dot product decays with distance)
sim_near = PE[5] @ PE[6]
sim_far = PE[5] @ PE[19]
assert sim_near > sim_far, 'nearby positions should be more similar'
print('Exercise 4 passed.')

## Exercise 5 — Causal mask: prove no peeking
Build a lower-triangular mask (True = allowed) and show that changing a FUTURE token never
changes the output at an EARLIER position. This is the property that makes autoregressive
generation valid.

In [ ]:
def causal_mask(seq):
    # TODO: (seq, seq) boolean, True on and below the diagonal.
    raise NotImplementedError

# --- tests ---
seq, d_k = 6, 8
m = causal_mask(seq)
assert m.shape == (seq, seq) and m.dtype == bool
assert m[2, 3] == False and m[3, 2] == True, 'position t may not see t+1'
Q = rng.normal(size=(seq, d_k)); K = rng.normal(size=(seq, d_k)); V = rng.normal(size=(seq, d_k))
out_a, _ = scaled_dot_product_attention(Q, K, V, mask=m)
V2 = V.copy(); V2[-1] += 100.0  # perturb the LAST (future) token only
out_b, _ = scaled_dot_product_attention(Q, K, V2, mask=m)
assert approx(out_a[:-1], out_b[:-1]), 'earlier positions must be unaffected by a future token'
print('Exercise 5 passed — causality holds.')

## Exercise 6 — OPTIONAL (torch): match the reference
Only after the above pass. Build the same projection with `torch.nn.MultiheadAttention`
(`batch_first=True`), copy your `Wq/Wk/Wv/Wo` into it, and confirm outputs match your numpy
version within tolerance. This is where you learn torch's weird in-projection packing.

In [ ]:
# OPTIONAL — requires: pip install torch
# import torch, torch.nn as nn
# torch packs Wq,Wk,Wv into one in_proj_weight of shape (3*d_model, d_model).
# Note: nn.MultiheadAttention applies x @ W.T, so transpose when copying your row-major Ws.
# TODO: instantiate nn.MultiheadAttention(d_model, h, bias=False, batch_first=True),
#       set in_proj_weight and out_proj.weight from your Ws, run, compare to mha(x).
print('Optional — implement if torch is installed.')

## Exercise 7 — OPTIONAL (nanoGPT): train and read the curve
Clone https://github.com/karpathy/nanoGPT, run the tiny-shakespeare char demo on CPU/GPU:
```
python data/shakespeare_char/prepare.py
python train.py config/train_shakespeare_char.py --device=cpu --compile=False \
    --max_iters=500 --eval_interval=100 --block_size=64 --batch_size=12
```
Record below: final **val loss**, and one generated sample. Then answer in `notes.md`:
*why does val loss plateau, and what would you change first to push it lower?*

In [ ]:
nanogpt_val_loss = None   # TODO: fill in after your run
nanogpt_sample = ''       # TODO: paste a short generated sample
print('Record your nanoGPT results here, then copy them into notes.md.')